# 1. Setup

Compare frozen, 2-block and 4-block versions of the same multi-path alignment model.

In [ ]:
import gc, json, math, sys
from pathlib import Path
from datetime import datetime
import timm, torch
import pandas as pd
from torch import nn
from timm.layers import set_reentrant_ckpt
from torch.nn import functional as F
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'common/dataset.py').is_file())
sys.path.insert(0, str(ROOT))
from common import config as cfg
from common.dataset import get_loaders, get_pairs, get_crop_and_disease_names, load_split, get_final_test_loaders
from common.evaluate import evaluate, val_accuracy, concept_accuracy
MODEL = 'vit_base_patch14_dinov2.lvd142m'
MODEL_NAME, FAMILY = 'approach3z_v2_depth_ablation', 'B'
DEBUG = True
HP = dict(backbone=MODEL, batch_size=8 if DEBUG else 16, epochs=1 if DEBUG else 20,
          unfreeze_blocks=[0, 2, 4], backbone_lr=1e-5,
          crop_lr=1e-4, disease_lr=3e-4, weight_decay=1e-4, label_smoothing=0.1,
          seed=cfg.SEED, norm='imagenet', hidden_dim=256, feature_dim=128,
          supcon_temperature=0.1, text_batch_size=16, dino_revision='main', text_revision='main',
          score_weights=[0.5,0.25,0.25],
          pair_templates=['a photo of a {crop} leaf with {disease}',
                          'a field photo of {crop} showing {disease} symptoms',
                          'a close-up leaf of {crop} affected by {disease}'],
          healthy_templates=['a photo of a healthy {crop} leaf','a field photo of a healthy {crop} plant'])
EPOCHS = HP['epochs']
cfg.NUM_WORKERS = 0  # Exact ToTnki runtime setting; common files are unchanged.
cfg.set_seed(cfg.SEED)
RUN = MODEL_NAME + ('_debug' if DEBUG else '') + '_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUT = cfg.CHECKPOINT_DIR / RUN
environment = dict(python=sys.version, torch=str(torch.__version__), timm=str(timm.__version__), device=str(cfg.DEVICE))

# 2. Data
Direct common loaders: formal7223 train/777 validation,323 development pairs. DEBUG uses common subsets. No extra holdout, sampler or augmentation. No official test CSV is read here.


In [ ]:
# Same common data path as the old CLIP notebook; DINO uses imagenet normalization.
frames = [load_split(cfg.TRAIN_CSV)]
assert len(frames[0]) == 8000
pairs = get_pairs()
assert pairs.shape == (cfg.NUM_SEEN_PAIRS, 2)
concept_names = get_crop_and_disease_names()
def loaders():
    cfg.set_seed(cfg.SEED)
    return get_loaders(HP['batch_size'], norm=HP['norm'], debug=DEBUG)
train_loader, val_loader, seen_loader, unseen_loader = loaders()
for loader in (train_loader, val_loader): loader.pin_memory = False
assert not set(train_loader.dataset.frame.image) & set(val_loader.dataset.frame.image)
sample_counts = dict(train=len(train_loader.dataset), val=len(val_loader.dataset))
print('Device:', cfg.DEVICE, 'Data:', cfg.DATA_DIR, 'Samples:', sample_counts)


# 3. Model

Only the number of unfrozen blocks changes.

In [ ]:
def snapshot(repository,revision,allow_download,patterns):
    from huggingface_hub import snapshot_download
    path=Path(snapshot_download(repository,revision=revision,allow_patterns=patterns,local_files_only=not allow_download))
    return path

class Dino(nn.Module):
    """Exact two-independent-tower initialization from ToTnki's released runner."""
    def __init__(self,config,device,allow_download=False,revision=None):
        super().__init__()
        import timm
        path=snapshot('timm/vit_base_patch14_dinov2.lvd142m',revision or config['dino_revision'],allow_download,
                      ['config.json','model.safetensors','pytorch_model.bin'])
        weights=next(path/name for name in ['model.safetensors','pytorch_model.bin'] if (path/name).is_file())
        def pinned_source_model(classes):
            # Same source factory, with only an explicit weight-file binding added.
            model=timm.create_model(MODEL,pretrained=True,num_classes=classes,img_size=cfg.IMAGE_SIZE,
                                    pretrained_cfg_overlay={'file':str(weights)})
            model.requires_grad_(False);model.head.requires_grad_(True)
            return model
        self.crop=pinned_source_model(cfg.NUM_CROPS)
        self.disease=pinned_source_model(cfg.NUM_DISEASES)
        self.augmentation_rng_state=torch.get_rng_state().clone()
        self.width=self.crop.num_features
        self.metadata=dict(repository='timm/vit_base_patch14_dinov2.lvd142m',revision=path.name,
            architecture=MODEL,width=self.width,independent_visual_towers=2,
            initialization='ToTnki source factory settings, crop then disease, explicitly pinned public weight file',
            preprocessing='common imagenet; img_size224; timm native position resampling and pre_logits')
        self.to(device)
    def policy(self, depth):
        self.requires_grad_(False).eval()
        for tower in (self.crop, self.disease):
            tower.set_grad_checkpointing(depth > 0)
            if depth > 0:
                tower.blocks[-depth:].requires_grad_(True)

    def features(self,pixels):
        return (self.crop.forward_head(self.crop.forward_features(pixels),pre_logits=True).float(),
                self.disease.forward_head(self.disease.forward_features(pixels),pre_logits=True).float())
    def forward(self,pixels):return self.features(pixels)

class TextOnly:
    """No CLIPModel, CLIPVisionModel, OpenCLIP model, or image encoder is created."""
    def __init__(self,config,device,allow_download=False,revision=None):
        from transformers import CLIPTextModelWithProjection, CLIPTokenizerFast
        path=snapshot('openai/clip-vit-base-patch16',revision or config['text_revision'],allow_download,
            ['*.json','vocab.json','merges.txt','model.safetensors','pytorch_model.bin'])
        self.model=CLIPTextModelWithProjection.from_pretrained(path,local_files_only=True).to(device).eval().requires_grad_(False)
        self.tokenizer=CLIPTokenizerFast.from_pretrained(path,local_files_only=True)
        if any('vision' in name for name,_ in self.model.named_modules()):raise RuntimeError('A CLIP vision encoder was instantiated.')
        self.dimension=self.model.config.projection_dim;self.device=device
        if self.dimension!=512:raise ValueError('Expected the CLIP B/16 512-dimensional text projection.')
        weight=next(path/name for name in ['model.safetensors','pytorch_model.bin'] if (path/name).is_file())
        self.metadata=dict(repository='openai/clip-vit-base-patch16',revision=path.name,
            class_name=type(self.model).__name__,dimension=self.dimension,trainable_parameters=0,clip_image_encoder=False)
    @torch.no_grad()
    def encode(self,prompts,size):
        self.assert_frozen();result=[]
        for start in range(0,len(prompts),size):
            tokens=self.tokenizer(prompts[start:start+size],padding=True,truncation=True,max_length=77,return_tensors='pt').to(self.device)
            result.append(F.normalize(self.model(**tokens).text_embeds.float(),dim=1))
        return torch.cat(result)
    def assert_frozen(self):
        if self.model.training or any(p.requires_grad or p.grad is not None for p in self.model.parameters()):
            raise RuntimeError('CLIP text encoder must be eval, frozen and gradient-free.')

def text_banks(text,pairs,names,config,ensemble=False):
    crops,diseases=names
    def clean(x):return x.replace('_',' ')
    pair_groups=[]
    for c,d in pairs.tolist():
        if clean(diseases[d]).strip().casefold()=='healthy':
            templates=config['healthy_templates'] if ensemble else config['healthy_templates'][:1]
        else:templates=config['pair_templates'] if ensemble else config['pair_templates'][:1]
        pair_groups.append([t.format(crop=clean(crops[c]),disease=clean(diseases[d])) for t in templates])
    flat=[p for group in pair_groups for p in group];encoded=text.encode(flat,config['text_batch_size'])
    bank=[];offset=0
    for group in pair_groups:
        bank.append(F.normalize(encoded[offset:offset+len(group)].mean(0),dim=0));offset+=len(group)
    crop_prompts=[f'a photo of a {clean(crops[i])} plant' for i in range(cfg.NUM_CROPS)]
    disease_prompts=[('a photo of a healthy leaf' if clean(diseases[i]).casefold()=='healthy' else f'a leaf showing {clean(diseases[i])}') for i in range(cfg.NUM_DISEASES)]
    return [torch.stack(bank),text.encode(crop_prompts,config['text_batch_size']),text.encode(disease_prompts,config['text_batch_size'])],dict(pair=pair_groups,crop=crop_prompts,disease=disease_prompts)

def projection(width,hidden,out):return nn.Sequential(nn.Linear(width,hidden),nn.ReLU(),nn.Linear(hidden,out))

def zscore(x):
    return (x-x.mean(1, keepdim=True)) / x.std(1, unbiased=False, keepdim=True).clamp_min(1e-6)

def factorized(parts, pairs, weights):
    pair, crop, disease = parts
    if len(weights) != 3 or min(weights) < 0 or abs(sum(weights)-1) > 1e-6:
        raise ValueError('Three nonnegative scorer weights summing to one required.')
    pairs = pairs.to(pair.device)
    return weights[0]*zscore(pair)+weights[1]*zscore(crop)[:, pairs[:,0]]+weights[2]*zscore(disease)[:, pairs[:,1]]


RECIPE = dict(pair=1., primitive=1.)


class Model(nn.Module):
    def __init__(self, backbone, banks):
        super().__init__()
        self.backbone = backbone
        width, hidden = backbone.width, HP['hidden_dim']
        dim = banks[0].shape[1]
        self.crop_projection = projection(width, hidden, dim)
        self.disease_projection = projection(width, hidden, dim)

        self.pair_projection = projection(width, hidden, dim)
        self.log_temperature = nn.Parameter(torch.tensor(math.log(1 / .07)))
        for index, bank in enumerate(banks):
            self.register_buffer(f'text{index}', bank)

    def install_text(self, banks):
        for index, bank in enumerate(banks):
            setattr(self, f'text{index}', bank)

    def forward(self, pixels):
        crop_features, disease_features = self.backbone(pixels)
        crop = F.normalize(self.crop_projection(crop_features), dim=1)
        disease = F.normalize(self.disease_projection(disease_features), dim=1)
        pair = F.normalize(self.pair_projection((crop_features + disease_features) / 2), dim=1)
        scale = self.log_temperature.clamp(0., math.log(100.)).exp()
        return scale * pair @ self.text0.T, scale * crop @ self.text1.T, scale * disease @ self.text2.T


def loss(output, labels):
    pair, crop, disease = output
    crops, diseases = pairs.to(labels.device)[labels].unbind(1)
    smoothing = HP['label_smoothing']
    primitive = (F.cross_entropy(crop, crops, label_smoothing=smoothing)
                 + F.cross_entropy(disease, diseases, label_smoothing=smoothing))
    return RECIPE['pair'] * F.cross_entropy(pair, labels, label_smoothing=smoothing) + RECIPE['primitive'] * primitive, 0.

def make_model(pairs, names, depth, metadata=None):
    cfg.set_seed(cfg.SEED)
    metadata = metadata or {}
    backbone = Dino(HP, cfg.DEVICE, allow_download=True, revision=metadata.get('dino', {}).get('revision'))
    backbone.policy(depth)
    text, banks = None, None

    text = TextOnly(HP, cfg.DEVICE, allow_download=True, revision=(metadata.get('text') or {}).get('revision'))
    banks, _ = text_banks(text, pairs, names, HP)

    torch.manual_seed(cfg.SEED)
    model = Model(backbone, banks).to(cfg.DEVICE)
    torch.set_rng_state(backbone.augmentation_rng_state)
    return model, text, dict(dino=backbone.metadata, text=text.metadata if text else None)


def make_predict(model, candidate_pairs):
    @torch.no_grad()
    def predict(images):
        model.eval()
        return factorized(model(images.to(cfg.DEVICE)), candidate_pairs, HP['score_weights'])
    return predict

# 4. Training

Save an independent validation winner for each depth, using the same pretrained revisions and seed.

In [ ]:
def train_trial(depth, revisions, best, checkpoint_path):
    train_loader, val_loader, _, _ = loaders()
    train_loader.pin_memory = val_loader.pin_memory = False
    model, text, metadata = make_model(pairs, concept_names, depth, revisions)
    del text  # Training uses the already computed frozen text embeddings.
    groups = {'backbone': [], 'crop': [], 'disease': []}
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            key = 'backbone' if name.startswith('backbone.') else 'disease' if 'disease' in name else 'crop'
            groups[key].append(parameter)
    optimizer = torch.optim.AdamW(
        [dict(params=groups[key], lr=HP[key + '_lr']) for key in groups if groups[key]],
        weight_decay=HP['weight_decay'])
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    predict = make_predict(model, pairs)
    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        model.backbone.eval()
        if depth > 0:
            for tower in (model.backbone.crop, model.backbone.disease):
                tower.blocks[-depth:].train()
        total_loss = anchor_count = samples = 0
        for images, _, _, labels in train_loader:
            optimizer.zero_grad(set_to_none=True)
            values = model(images.to(cfg.DEVICE))
            value, anchor_fraction = loss(values, labels.to(cfg.DEVICE))
            if not torch.isfinite(value):
                raise RuntimeError('Nonfinite training loss.')
            value.backward()
            optimizer.step()
            samples += len(labels)
            total_loss += value.item() * len(labels)
            anchor_count += anchor_fraction * len(labels)
        schedule.step()
        validation = val_accuracy(predict, val_loader)
        record = dict(depth=depth, epoch=epoch, val_top1=validation, loss=total_loss / samples)
        if FAMILY == 'A':
            record['valid_crosscrop_anchor_fraction'] = anchor_count / samples
        history.append(record)
        print(record)
        if best is None or validation > best['val_top1']:
            best = record
            torch.save(dict(identity, depth=depth, best_val=validation, best_epoch=epoch, pairs=pairs,
                            metadata=metadata, state_dict=model.state_dict()), checkpoint_path)
    return best, history, metadata


OUT.mkdir(parents=True, exist_ok=False)
set_reentrant_ckpt(False)
identity = dict(model_name=MODEL_NAME, family=FAMILY, debug=DEBUG, hp=HP, recipe=RECIPE)
revisions, trials = None, []
for depth in HP['unfreeze_blocks']:
    folder = OUT / f'depth_{depth}'
    folder.mkdir()
    path = folder / (MODEL_NAME + '.pt')
    best, history, revisions = train_trial(depth, revisions, None, path)
    selection = dict(identity, checkpoint=path.name, depth=depth, best_val=best['val_top1'],
                     best_epoch=best['epoch'], history=history,
                     train_images=train_loader.dataset.frame.image.tolist(),
                     validation_images=val_loader.dataset.frame.image.tolist(),
                     development_pairs=pairs.tolist())
    (folder / 'selection.json').write_text(json.dumps(selection, indent=2), encoding='utf8')
    trials.append(best)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
display(pd.DataFrame(trials))

# 5. Evaluation

Evaluate all three frozen checkpoint choices with shared metrics.

In [ ]:
RUN_FINAL_EVALUATION = False  # Enable only after all model choices are frozen.
reports = []
if RUN_FINAL_EVALUATION:
    if DEBUG:
        raise ValueError('Official evaluation requires a full non-DEBUG run.')
    for depth in HP['unfreeze_blocks']:
        if not (OUT / f'depth_{depth}' / 'selection.json').is_file():
            raise ValueError('Complete all depth trials before official evaluation.')
    for depth in HP['unfreeze_blocks']:
        run_folder = OUT / f"depth_{depth}"
        selection = json.loads((run_folder / 'selection.json').read_text(encoding='utf8'))
        checkpoint = torch.load(run_folder / selection['checkpoint'], map_location='cpu', weights_only=True)
        expected = dict(model_name=MODEL_NAME, family=FAMILY, debug=False, hp=HP, recipe=RECIPE)
        if any(record.get(key) != value for record in (selection, checkpoint)
               for key, value in expected.items()):
            raise ValueError('The selected run or checkpoint does not match the frozen settings.')
        if selection['development_pairs'] != pairs.tolist() or not torch.equal(checkpoint['pairs'], pairs):
            raise ValueError('Development candidates changed.')
        if (selection['train_images'] != train_loader.dataset.frame.image.tolist()
                or selection['validation_images'] != val_loader.dataset.frame.image.tolist()):
            raise ValueError('Common train/validation membership changed.')
        if checkpoint['depth'] != depth or (checkpoint['depth'], checkpoint['best_epoch'], checkpoint['best_val']) != (
                selection['depth'], selection['best_epoch'], selection['best_val']):
            raise ValueError('Checkpoint does not match the validation winner.')
        model, text, metadata = make_model(pairs, concept_names, checkpoint['depth'], checkpoint['metadata'])
        if metadata != checkpoint['metadata']:
            raise ValueError('Public pretrained model identity changed.')
        model.load_state_dict(checkpoint['state_dict'])
        model.eval().requires_grad_(False)
        for tower in (model.backbone.crop, model.backbone.disease):
            tower.set_grad_checkpointing(False)
        # Only now open the official candidates and test loaders.
        final_pairs = get_pairs(final_evaluation=True)
        final_banks, _ = text_banks(text, final_pairs, concept_names, HP)
        model.install_text(final_banks)
        seen_loader, unseen_loader = get_final_test_loaders(
            HP['batch_size'], norm=HP['norm'], final_evaluation=True)
        sample_counts.update(seen=len(seen_loader.dataset), unseen=len(unseen_loader.dataset))
        predict_final = make_predict(model, final_pairs)
        # common.evaluate predicts clean seen, then clean unseen, then robustness.
        seen_count = sample_counts['seen']
        clean_end = seen_count + sample_counts['unseen']
        unseen_batches, processed = [], 0
        def predict(images):
            global processed
            scores = predict_final(images)
            if seen_count <= processed < clean_end:
                unseen_batches.append(scores.float().cpu())
            processed += len(images)
            return scores
        RESULTS = cfg.RESULTS_DIR / OUT.name / run_folder.name
        RESULTS.mkdir(parents=True, exist_ok=False)
        previous_results = cfg.RESULTS_DIR
        try:
            cfg.RESULTS_DIR = RESULTS
            result = evaluate(predict, seen_loader, unseen_loader, MODEL_NAME, dict(HP, selected_depth=checkpoint['depth']), robust=True)
        finally:
            cfg.RESULTS_DIR = previous_results
        unseen_labels = torch.as_tensor(unseen_loader.dataset.frame.pair_row.to_numpy())
        crop_acc, disease_acc = concept_accuracy(torch.cat(unseen_batches), unseen_labels, final_evaluation=True)
        result.update(unseen_test_crop_acc=crop_acc, unseen_test_disease_acc=disease_acc,
                      depth=depth, debug=DEBUG, best_val=checkpoint['best_val'])
        (RESULTS / f"{MODEL_NAME}.json").write_text(json.dumps(result, indent=2), encoding="utf8")
        reports.append(result)
        del predict_final, model, text, unseen_batches

# 6. Save

Write the depth comparison.

In [ ]:
if reports:
    table = pd.DataFrame(reports)[['depth', 'seen_top1', 'seen_top5', 'unseen_top1', 'unseen_top5', 'harmonic_mean']]
    table.to_csv(cfg.RESULTS_DIR / OUT.name / 'depth_comparison.csv', index=False)
    display(table)
else:
    print('Checkpoint folders:', OUT)